# Final Model Selection

This notebook selects one final candidate using the existing experiment artifacts. It does not retrain or reevaluate either model, change data or preprocessing, or create new visualizations.

In [1]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else (cwd / 'pixels-to-water' if (cwd / 'pixels-to-water' / 'results').exists() else cwd)
RESULTS_DIR = PROJECT_ROOT / 'results'
OUTPUT_PATH = RESULTS_DIR / 'final_model_selection.json'
print(f'Project root: {PROJECT_ROOT}')

Project root: c:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water


In [2]:
comparison = pd.read_csv(RESULTS_DIR / 'model_comparison.csv')
comparison = comparison.set_index('Metric')
test_manifest = pd.read_csv(RESULTS_DIR / 'test.csv')
error_summary = pd.read_csv(RESULTS_DIR / 'error_analysis_summary.csv')

metric_names = ['Test Accuracy', 'Precision', 'Recall', 'F1-score', 'Top-5 Accuracy']
key_metrics = comparison.loc[metric_names, ['ResNet-18', 'EfficientNet-B0']].to_dict()
print(comparison.loc[metric_names, ['ResNet-18', 'EfficientNet-B0']].to_string())
print(f'Test set size: {len(test_manifest)}')

                ResNet-18  EfficientNet-B0
Metric                                    
Test Accuracy    0.686000         0.742000
Precision        0.725493         0.779000
Recall           0.686000         0.742000
F1-score         0.679440         0.737577
Top-5 Accuracy   0.904000         0.928000
Test set size: 500


In [3]:
error_counts = error_summary.groupby('model')['number_of_errors'].sum().to_dict()
cultivars = set(test_manifest['cultivar'])
errors_by_cultivar = error_summary.groupby(['model', 'true_cultivar'])['number_of_errors'].sum().unstack(fill_value=0)
for model in ['ResNet-18', 'EfficientNet-B0']:
    if model not in errors_by_cultivar.index:
        errors_by_cultivar.loc[model] = 0
errors_by_cultivar = errors_by_cultivar.reindex(columns=sorted(cultivars), fill_value=0)
resnet_errors = errors_by_cultivar.loc['ResNet-18']
efficientnet_errors = errors_by_cultivar.loc['EfficientNet-B0']
error_analysis = {
    'ResNet-18 errors': int(error_counts['ResNet-18']),
    'EfficientNet-B0 errors': int(error_counts['EfficientNet-B0']),
    'EfficientNet-B0 fewer errors by': int(error_counts['ResNet-18'] - error_counts['EfficientNet-B0']),
    'cultivars where ResNet-18 had fewer errors': int((resnet_errors < efficientnet_errors).sum()),
    'cultivars where EfficientNet-B0 had fewer errors': int((efficientnet_errors < resnet_errors).sum()),
    'cultivars tied': int((resnet_errors == efficientnet_errors).sum()),
}
print(pd.DataFrame([error_analysis]).to_string(index=False))

 ResNet-18 errors  EfficientNet-B0 errors  EfficientNet-B0 fewer errors by  cultivars where ResNet-18 had fewer errors  cultivars where EfficientNet-B0 had fewer errors  cultivars tied
              157                     129                               28                                          24                                                37              39


In [6]:
selected_model = 'EfficientNet-B0'
reason = (
    'Based on the experiments conducted in this project, EfficientNet-B0 is selected as the final candidate '
    'because it demonstrated the strongest overall performance under the evaluated conditions. It had the '
    'higher test accuracy, precision, recall, F1-score, and top-5 accuracy, and the existing error analysis '
    'recorded 28 fewer test-set errors.'
)
selection = {
    'selected_model': selected_model,
    'reason': reason,
    'key_metrics': key_metrics,
    'statistical_reliability_result': statistical_reliability_result,
}
with open(OUTPUT_PATH, 'w', encoding='utf-8') as file:
    json.dump(selection, file, indent=2)

print(f'Final selected model: {selected_model}')
print(reason)
print(f'Saved: {OUTPUT_PATH}')

Final selected model: EfficientNet-B0
Based on the experiments conducted in this project, EfficientNet-B0 is selected as the final candidate because it demonstrated the strongest overall performance under the evaluated conditions. It had the higher test accuracy, precision, recall, F1-score, and top-5 accuracy, and the existing error analysis recorded 28 fewer test-set errors.
Saved: c:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\results\final_model_selection.json
